# FMA Small: initial data exploration

Data Loading and Exploration: audit precomputed audio features and preserve all original rows and columns. No imputation, scaling, feature removal, PCA, similarity modeling, or audio decoding occurs here.

Run all cells with the project `.venv` kernel from either the repository root or `notebooks/`. Reusable diagnostics live in `src/exploration.py`; the original FMA `features.py`, `utils.py`, and `usage.ipynb` remain references from the fma repo itself

In [ ]:
from pathlib import Path
import sys
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data/fma_metadata/tracks.csv").is_file()
             and (p / "src/exploration.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the repository root or notebooks directory; see README.md")
sys.path.insert(0, str(ROOT / "src"))
from exploration import load_small, quality_tables, distribution_table, correlation_pairs, readable_table

REPORTS = ROOT / "reports"
REPORTS.mkdir(exist_ok=True)
pd.set_option("display.max_rows", 25)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", None)
plt.rcParams.update({"figure.figsize": (12, 4), "figure.dpi": 110})
print("Repository:", ROOT.name)
print({name: version(name) for name in ["pandas", "numpy", "matplotlib", "nbformat", "nbclient"]})

## Load and align by track ID

The supplied CSVs cover the full dataset, so select `('set', 'subset') == 'small'` before auditing features. Explicit ID selection preserves metadata order. Duplicate/missing/invalid IDs, duplicate column labels, missing Small feature rows, or nonnumeric feature columns stop loading with an error.

For readable tables, flatten each feature tuple to **family__statistic__component**, e.g. `mfcc__mean__01`. `mean` summarizes audio frames within one track; `01` identifies the coefficient/band/bin within that family, not the track ID. The component's interpretation depends on the family. The original MultiIndex remains available for grouped analysis; `features_flat` provides a flat copy for exploration. Flattening changes labels only.

In the distribution table, the row `mfcc__mean__01` identifies the stored per-track summary, whereas the table's `mean` and `std` columns describe how that summary varies **across tracks**.

In [ ]:
metadata, features, inventory = load_small(ROOT / "data/fma_metadata")
assert features.index.equals(metadata.index)
assert metadata[("set", "subset")].eq("small").all()
features_flat = readable_table(features)
feature_dictionary = features.columns.to_frame(index=False)
feature_dictionary.columns = ["family", "within_track_statistic", "component"]
feature_dictionary.insert(0, "feature_name", features_flat.columns)
feature_dictionary.to_csv(REPORTS / "feature_dictionary.csv", index=False)
# Confirm flattening changes neither order, values, nor track IDs.
assert features_flat.index.equals(features.index)
np.testing.assert_array_equal(features_flat.to_numpy(), features.to_numpy())
display(inventory.to_frame())
display(features.dtypes.astype(str).value_counts().rename_axis("dtype").to_frame("columns"))
display(readable_table(metadata.loc[:, [("track", "title"), ("artist", "name"), ("track", "genre_top")]].head()))
print("Feature label guide (complete guide saved to reports/feature_dictionary.csv):")
display(feature_dictionary.head(8))
print("Preview: feature names down rows, track IDs across columns")
display(features_flat.iloc[:5, :8].T)
print("Statistics:", sorted(features.columns.get_level_values(1).unique()))

In [ ]:
groups = features.T.groupby(level=0, sort=True).size().rename("columns")
display(groups.to_frame())
display(metadata[("track", "genre_top")].value_counts().rename("tracks").to_frame())
groups.plot.bar(title="Precomputed feature columns by family", ylabel="Columns", rot=35)
plt.tight_layout()
plt.show()

## Audio directory structure only

The observed layout is `data/fma_small/<first three digits>/<six-digit track ID>.mp3`, for example `data/fma_small/000/000010.mp3`, matching `utils.get_audio_path`. The root contains 156 numbered directories plus `README.txt`, `checksums`, and `.DS_Store` at inspection time. Only the root and three example entries in one directory were inspected. No recursive audio listing, full existence check, checksum validation, or audio decoding was performed; file completeness and readability remain unverified. The following cell only constructs example paths from track IDs.

In [ ]:
example_ids = metadata.index[:5]
def audio_relative_path(track_id):
    padded = f"{int(track_id):06d}"
    return f"data/fma_small/{padded[:3]}/{padded}.mp3"
display(pd.DataFrame({"track_id": example_ids,
                      "expected_relative_path": [audio_relative_path(t) for t in example_ids]}))

## Missing values, infinities, and constant columns

Displayed/exported feature tables use flat feature names; track tables retain track IDs. NaN and positive/negative infinity are counted separately, with percentages for every feature and track. `finite` is a temporary diagnostic view that treats infinities as missing for statistics; it does not modify or replace `features`. A constant column has exactly one unique finite value; an all-invalid column has none.

In [ ]:
finite, feature_quality, track_quality = quality_tables(features)
constant = feature_quality.loc[feature_quality.finite_unique.eq(1)]
all_invalid = feature_quality.loc[feature_quality.finite_unique.eq(0)]
quality_summary = pd.Series({
    "nan_cells": int(feature_quality.nan.sum()),
    "positive_inf_cells": int(feature_quality.positive_inf.sum()),
    "negative_inf_cells": int(feature_quality.negative_inf.sum()),
    "affected_tracks": int(track_quality.nonfinite.gt(0).sum()),
    "affected_features": int(feature_quality.nonfinite.gt(0).sum()),
    "constant_features": len(constant), "all_invalid_features": len(all_invalid),
})
display(quality_summary.to_frame("count"))
display(readable_table(feature_quality.sort_values("nonfinite", ascending=False).head(15)))
display(track_quality.sort_values("nonfinite", ascending=False).head(15))
display(readable_table(constant))
readable_table(feature_quality).to_csv(REPORTS / "feature_quality.csv")
track_quality.to_csv(REPORTS / "track_quality.csv")

## Duplicate vectors and potentially invalid summary values

Distinct IDs can have exactly equal numeric feature vectors. These are investigation candidates, not proof that recordings are duplicates. The report lists all involved IDs and descriptive metadata. Hashes label groups after exact duplicate detection, and each group is checked for exact equality. Basic summary consistency checks flag negative standard deviations and means/medians outside minimum/maximum bounds; they do not establish audio validity.

In [ ]:
duplicate_extra = int(features.duplicated().sum())
duplicate_mask = features.duplicated(keep=False)
duplicate_features = features.loc[duplicate_mask]
duplicate_details = metadata.loc[duplicate_mask, [("track", "title"), ("artist", "name"), ("track", "genre_top")]].copy()
duplicate_details.columns = ["title", "artist", "genre_top"]
if len(duplicate_details):
    hashes = pd.util.hash_pandas_object(duplicate_features, index=False)
    duplicate_details["vector_group"] = pd.factorize(hashes)[0] + 1
    for _, ids in duplicate_details.groupby("vector_group").groups.items():
        values = features.loc[ids].to_numpy()
        assert np.array_equal(values, np.broadcast_to(values[0], values.shape), equal_nan=True)
print("Repeated vectors beyond first occurrence:", duplicate_extra)
print("All tracks involved:", int(duplicate_mask.sum()))
display(duplicate_details)
duplicate_details.to_csv(REPORTS / "duplicate_vectors.csv")

checks = {}
for family in features.columns.get_level_values(0).unique():
    block = finite[family]
    checks[f"{family}: negative std"] = int(block["std"].lt(0).sum().sum())
    for stat in ["mean", "median"]:
        outside = block[stat].lt(block["min"] - 1e-8) | block[stat].gt(block["max"] + 1e-8)
        checks[f"{family}: {stat} outside min/max"] = int(outside.sum().sum())
consistency = pd.Series(checks, name="flagged_cells")
display(consistency.to_frame())

## Distributions, scale differences, skew, and outlier flags

Statistics use finite observations. The complete CSV contains counts, quantiles, standard deviations, skew, excess kurtosis, and the fraction beyond three IQRs. Absolute skew above 2 is a screening heuristic. A modal fraction of at least 99% flags nearly invariant columns even if their variance is nonzero. Neither heuristic automatically removes features or tracks. With zero IQR, even a small deviation may be flagged. Histogram axes show raw values; no transformations are fitted.

In [ ]:
distributions = distribution_table(finite)
readable_table(distributions).to_csv(REPORTS / "feature_distributions.csv")
strong_skew = distributions.loc[distributions["skew"].abs().gt(2)]
near_invariant = distributions.loc[distributions.modal_fraction.ge(.99)]
print("Features with |skew| > 2:", len(strong_skew))
print("Features with modal fraction >= 99%:", len(near_invariant))
display(readable_table(distributions.sort_values("skew", key=lambda x: x.abs(), ascending=False).head(15)))
display(readable_table(near_invariant[["std", "min", "max", "modal_fraction", "skew"]]))
display(distributions[["std", "min", "max"]].agg(["min", "median", "max"]))
selected = [("mfcc", "mean", "01"), ("spectral_centroid", "mean", "01"),
            ("rmse", "mean", "01"), ("chroma_stft", "max", "01"),
            ("chroma_stft", "kurtosis", "01"), ("tonnetz", "mean", "01")]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, column in zip(axes.flat, selected):
    ax.hist(finite[column].dropna(), bins=60)
    ax.set_title("__".join(column), fontsize=10)
    ax.set_ylabel("Tracks")
fig.suptitle("Raw feature distributions, including long tails and saturated maxima")
fig.tight_layout()
plt.show()
fig, ax = plt.subplots(figsize=(12, 4))
for family, subset in distributions.groupby(level=0):
    y = subset["std"]
    ax.scatter([family] * len(y), y.where(y.gt(0)), s=10, alpha=.45)
ax.set_yscale("log")
ax.set_ylabel("Feature standard deviation (log axis)")
ax.set_title("Scale varies across and within feature families")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()

In [ ]:
correlations, high_pairs = correlation_pairs(finite, threshold=.95)
high_pairs.to_csv(REPORTS / "high_correlations.csv", index=False)
print("Unique pairs with |Pearson r| >= 0.95 (no self-pairs):", len(high_pairs))
display(high_pairs.groupby("relationship").size().to_frame("pairs"))
for relationship in ["same component, different statistic",
                     "different components, same family", "different families"]:
    print(relationship)
    display(high_pairs.loc[high_pairs.relationship.eq(relationship)].head(10))

# Scatterplots show that observations are tracks, and label both full columns.
examples = []
for relationship in ["same component, different statistic", "different families"]:
    candidates = high_pairs.loc[high_pairs.relationship.eq(relationship)]
    if not candidates.empty:
        examples.append(candidates.iloc[0])
if examples:
    fig, axes = plt.subplots(1, len(examples), figsize=(7 * len(examples), 5), squeeze=False)
    for ax, pair in zip(axes.flat, examples):
        x, y = features_flat[pair.feature_a], features_flat[pair.feature_b]
        valid = np.isfinite(x) & np.isfinite(y)
        ax.scatter(x[valid], y[valid], s=7, alpha=.15)
        ax.set_xlabel(pair.feature_a)
        ax.set_ylabel(pair.feature_b)
        ax.set_title(f"{pair.relationship}\nr = {pair.pearson_r:.4f}; n = {pair.paired_tracks:,}")
    fig.suptitle("Each point represents one track")
    fig.tight_layout()
    plt.show()

fig, ax = plt.subplots(figsize=(9, 8))
heatmap = correlations.to_numpy().copy()
np.fill_diagonal(heatmap, np.nan)
im = ax.imshow(heatmap, vmin=-1, vmax=1, cmap="coolwarm", interpolation="nearest")
labels = features.columns.get_level_values(0)
families = list(dict.fromkeys(labels))
centers = [np.flatnonzero(labels == family).mean() for family in families]
ax.set_xticks(centers, families, rotation=90, fontsize=8)
ax.set_yticks(centers, families, fontsize=8)
ax.set_title(f"Pearson correlation: {features.shape[1]} columns (self-pairs hidden)")
fig.colorbar(im, ax=ax, label="Pearson r", shrink=.75)
fig.tight_layout()
plt.show()

## Findings and next-phase recommendations

This cell writes the concise report from the values computed above. The diagnostic CSVs retain every feature/track, even when displayed tables are abbreviated. No cleaned/scaled/reduced matrix is produced. Audio readability and completeness are outside this audit.

In [ ]:
report = f"""# FMA Small exploration findings

## Scope and alignment

- Full metadata: {inventory['metadata_tracks']:,} tracks; full feature table: {inventory['all_feature_tracks']:,} tracks.
- Small subset: {len(features):,} unique positive track IDs, aligned with {features.shape[1]} numerical audio columns in {len(groups)} families and seven summary statistics.
- Missing Small feature rows: {inventory['small_missing_feature_rows']}; duplicate metadata/feature IDs: 0/0. No unmatched IDs across the full metadata and feature tables.
- Genre labels contain eight groups of 1,000 tracks each; they remain metadata and are excluded from the numeric matrix.
- Families (column counts): {', '.join(f'{name} ({count})' for name, count in groups.items())}.

## Data quality and distributions

- NaNs: {quality_summary['nan_cells']}; positive infinity: {quality_summary['positive_inf_cells']}; negative infinity: {quality_summary['negative_inf_cells']}.
- Constant columns: {len(constant)}; entirely invalid columns: {len(all_invalid)}.
- Exact repeated numeric vectors beyond the first: {duplicate_extra}; all involved tracks: {int(duplicate_mask.sum())}. See duplicate_vectors.csv for IDs, groups, and metadata. Distinct IDs alone do not prove that the audio is distinct.
- Basic summary consistency violations: {int(consistency.sum())} cells (negative standard deviations or means/medians outside min/max, tolerance 1e-8).
- Absolute skew above 2: {len(strong_skew)} of {features.shape[1]} columns. This includes extreme kurtosis tails and highly concentrated chroma maxima; skew is a diagnostic, not evidence of corrupt recordings.
- At least 99% of finite observations share one value in {len(near_invariant)} columns. Inspect these for saturation and limited information before deciding whether to remove them.
- Feature standard deviations range from {distributions['std'].min():.6g} to {distributions['std'].max():.6g}; raw scales differ substantially.
- Unique feature pairs with absolute Pearson correlation >= 0.95: {len(high_pairs)}. Relationship counts: {', '.join(f'{name}: {count}' for name, count in high_pairs.relationship.value_counts().items())}. Correlation is computed across tracks between distinct family/statistic/component columns; mean-versus-median comparisons of one descriptor test summary redundancy. Self-pairs are excluded. See high_correlations.csv; the count is not the number of removable columns.

## Recommended decisions for later phases

1. No missing-value imputation or constant-column removal is indicated by this Small-subset audit. Retain the raw numeric matrix and its track ID index unchanged.
2. Investigate equal-vector track groups using titles, artists, and later targeted audio checks before deduplicating; do not automatically discard songs.
3. Review nearly invariant columns and long-tailed summaries individually. Preserve unusual songs; investigate saturation and robust or skew-reducing transformations rather than blanket outlier deletion. Signed MFCC/tonnetz/skew values rule out a universal logarithm.
4. Compare StandardScaler as a baseline with robust scaling or suitable per-column transforms after reviewing distributions. Fit transformations on training data only if evaluating held-out results, then apply the same fitted transformation to validation/test data.
5. Explore correlation filtering at 0.95 with an explicit deterministic policy that preserves interpretable summaries. Check sensitivity to outliers or rank correlation before removal.
6. Subsequently compare PCA at 95% explained variance against cleaned/scaled originals and correlation-filtered features. PCA is not performed here and is not assumed preferable.
7. Metadata, including genre/artist/title, stays separate from acoustic features. Record feature order and ID alignment with any future saved matrices and fitted transformations.

## Audio structure and limits

Expected path: data/fma_small/<first-three-digits>/<six-digit-track-id>.mp3. The initial shallow inspection found 156 numbered directories and sampled three entries from directory 000. No recursive audio enumeration, audio decoding, full existence checks, or readability checks were performed. Sampling-rate changes are deferred because this phase uses existing features.

## Reproduction and artifacts

Run all cells in notebooks/explore_fma.ipynb with the project environment. Feature names use family__statistic__component; feature_dictionary.csv maps labels to the original levels. CSV files in reports/ contain complete per-feature/per-track quality counts, descriptive statistics, duplicate-vector groups, and correlation candidates. Percentages use all tracks/columns for missingness, and finite observations for distribution diagnostics. No raw files or precomputed feature values were changed.
"""
(REPORTS / "fma_small_exploration.md").write_text(report)
# from IPython.display import Markdown
# display(Markdown(report))